# D0 Temporary Duration Analysis

Temporary notebook for evaluating planned duration benchmark variables before any runtime implementation.

Scope:
- Completed trials first.
- Historical total duration target: `COMPLETED + completion_date_type == ACTUAL + completion_duration_months > 0`.
- Historical primary readout context: `COMPLETED + primary_completion_date_type == ACTUAL + primary_completion_duration_months > 0`.
- Non-completed `ESTIMATED` dates are counted only as future planned-candidate evidence, not used in the completed-only benchmark target here.
- Non-completed `ACTUAL` dates are lower-bound or early-stop context only.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'data_clinpred.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PATH = PROJECT_ROOT / 'data' / 'data_clinpred.csv'
MIN_N = 50

CANDIDATES = [
    'phase_ml', 'therapeutic_area_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml',
    'therapeutic_modality_ml', 'sponsor_tier_ml', 'administration_complexity_ml',
    'endpoint_rigor_ml', 'endpoint_structure_ml', 'intervention_model_ml', 'allocation_ml',
    'masking_ml', 'comparator_benchmark_ml', 'has_placebo_ml', 'healthy_volunteers_ml',
    'adult_ml', 'child_ml', 'older_adult_ml', 'includes_us_ml', 'patient_severity_ml',
    'line_of_therapy_ml', 'number_of_arms_ml', 'primary_duration_months_ml',
]

cols = list(dict.fromkeys([
    'nct_id', 'overall_status', 'completion_date_type', 'primary_completion_date_type',
    'completion_duration_months', 'primary_completion_duration_months',
    'phase_ml', 'therapeutic_area_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml',
] + CANDIDATES))

df = pd.read_csv(PATH, usecols=cols, low_memory=False)
for col in ['completion_duration_months', 'primary_completion_duration_months', 'primary_duration_months_ml', 'number_of_arms_ml']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['endpoint_bin'] = pd.cut(
    df['primary_duration_months_ml'],
    [-np.inf, 3, 6, 12, 18, 24, 36, 60, np.inf],
    labels=['<=3', '3-6', '6-12', '12-18', '18-24', '24-36', '36-60', '>60'],
).astype('object').where(df['primary_duration_months_ml'].notna(), 'NA')

completed = df['overall_status'].eq('COMPLETED')
print('rows', len(df))
print('completed rows', int(completed.sum()))
print(df[['overall_status', 'completion_date_type', 'primary_completion_date_type']].describe(include='all'))


In [ ]:
def duration_summary(mask: pd.Series, value_col: str) -> dict[str, float | int]:
    s = df.loc[mask, value_col].dropna()
    qs = s.quantile([.01, .05, .10, .25, .50, .75, .90, .95, .99])
    return {
        'n': int(len(s)),
        'mean': round(float(s.mean()), 2),
        'p01': round(float(qs.loc[.01]), 2),
        'p05': round(float(qs.loc[.05]), 2),
        'p10': round(float(qs.loc[.10]), 2),
        'p25': round(float(qs.loc[.25]), 2),
        'p50': round(float(qs.loc[.50]), 2),
        'p75': round(float(qs.loc[.75]), 2),
        'p90': round(float(qs.loc[.90]), 2),
        'p95': round(float(qs.loc[.95]), 2),
        'p99': round(float(qs.loc[.99]), 2),
        'min': round(float(s.min()), 2),
        'max': round(float(s.max()), 2),
        'lt1': int((s < 1).sum()),
        'lt3': int((s < 3).sum()),
        'gt120': int((s > 120).sum()),
    }

total_target = completed & df['completion_date_type'].eq('ACTUAL') & df['completion_duration_months'].gt(0)
primary_target = completed & df['primary_completion_date_type'].eq('ACTUAL') & df['primary_completion_duration_months'].gt(0)

pd.DataFrame([
    {'target': 'total_completion', **duration_summary(total_target, 'completion_duration_months')},
    {'target': 'primary_readout', **duration_summary(primary_target, 'primary_completion_duration_months')},
])


In [ ]:
def group_stats(base: pd.DataFrame, ycol: str, keys: list[str], min_n: int = MIN_N) -> dict[str, float | int]:
    grouped = base.groupby(keys, dropna=False)[ycol].agg(['size', 'median'])
    usable = grouped[grouped['size'] >= min_n]
    eligible = base.set_index(keys).index.isin(usable.index)
    return {
        'groups': int(len(grouped)),
        'groups_ge50': int(len(usable)),
        'rows_ge50': int(eligible.sum()),
        'coverage': round(float(eligible.mean()), 4),
    }

hierarchies = {
    'phase_only': ['phase_ml'],
    'phase_ta': ['phase_ml', 'therapeutic_area_ml'],
    'phase_ta_rare': ['phase_ml', 'therapeutic_area_ml', 'is_rare_disease_ml'],
    'phase_indication_rare': ['phase_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml'],
    'phase_indication_rare_modality': ['phase_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml', 'therapeutic_modality_ml'],
    'phase_ta_modality': ['phase_ml', 'therapeutic_area_ml', 'therapeutic_modality_ml'],
    'clinical_endpoint_bin': ['phase_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml', 'endpoint_bin'],
}

rows = []
for label, mask, ycol in [
    ('total_completion', total_target, 'completion_duration_months'),
    ('primary_readout', primary_target, 'primary_completion_duration_months'),
]:
    base = df.loc[mask].copy()
    for name, keys in hierarchies.items():
        rows.append({'target': label, 'level': name, **group_stats(base, ycol, keys)})

pd.DataFrame(rows)


In [ ]:
def clean_cat(series: pd.Series) -> pd.Series:
    return series.astype('object').where(pd.notna(series), 'NA').astype(str)


def binned(name: str, series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    if name == 'primary_duration_months_ml':
        return pd.cut(values, [-np.inf, 3, 6, 12, 18, 24, 36, 60, np.inf], labels=['<=3', '3-6', '6-12', '12-18', '18-24', '24-36', '36-60', '>60'])
    if name == 'number_of_arms_ml':
        return pd.cut(values, [-np.inf, 1, 2, 3, 4, np.inf], labels=['1', '2', '3', '4', '5+'])
    return series


def eta_squared(cat: pd.Series, ylog: pd.Series) -> float:
    sample = pd.DataFrame({'cat': clean_cat(cat), 'y': ylog}).dropna()
    if sample.empty or sample['cat'].nunique() < 2:
        return np.nan
    overall = sample['y'].mean()
    total = ((sample['y'] - overall) ** 2).sum()
    grouped = sample.groupby('cat')['y'].agg(['mean', 'size'])
    between = (grouped['size'] * ((grouped['mean'] - overall) ** 2)).sum()
    return float(between / total) if total > 0 else 0.0


def candidate_effects(mask: pd.Series, ycol: str) -> pd.DataFrame:
    base = df.loc[mask].copy()
    ylog = np.log1p(base[ycol])
    rows = []
    for col in CANDIDATES:
        cat = binned(col, base[col])
        sample = pd.DataFrame({'cat': clean_cat(cat), 'y': base[ycol]}).dropna()
        counts = sample['cat'].value_counts()
        medians = sample.groupby('cat')['y'].median()
        rows.append({
            'field': col,
            'eta2_log': round(eta_squared(cat, ylog), 4),
            'unique': int(sample['cat'].nunique()),
            'min_n': int(counts.min()),
            'groups_ge50': int((counts >= MIN_N).sum()),
            'largest_n': int(counts.max()),
            'median_min': round(float(medians.min()), 2),
            'median_max': round(float(medians.max()), 2),
            'median_range': round(float(medians.max() - medians.min()), 2),
        })
    return pd.DataFrame(rows).sort_values(['eta2_log', 'groups_ge50'], ascending=[False, False])

candidate_effects(total_target, 'completion_duration_months').head(18)


In [ ]:
candidate_effects(primary_target, 'primary_completion_duration_months').head(18)


In [ ]:
variants = {
    'global': [[]],
    'phase_only': [['phase_ml']],
    'clinical_hierarchy': [
        ['phase_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml'],
        ['phase_ml', 'therapeutic_area_ml', 'is_rare_disease_ml'],
        ['phase_ml', 'therapeutic_area_ml'],
        ['phase_ml'],
    ],
    'clinical_then_ta_modality': [
        ['phase_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml'],
        ['phase_ml', 'therapeutic_area_ml', 'is_rare_disease_ml'],
        ['phase_ml', 'therapeutic_area_ml', 'therapeutic_modality_ml'],
        ['phase_ml', 'therapeutic_area_ml'],
        ['phase_ml'],
    ],
    'clinical_endpoint_bin': [
        ['phase_ml', 'gbd_cause_id_3_ml', 'is_rare_disease_ml', 'endpoint_bin'],
        ['phase_ml', 'therapeutic_area_ml', 'is_rare_disease_ml', 'endpoint_bin'],
        ['phase_ml', 'therapeutic_area_ml', 'endpoint_bin'],
        ['phase_ml', 'endpoint_bin'],
        ['phase_ml'],
    ],
    'endpoint_then_clinical': [
        ['phase_ml', 'endpoint_bin'],
        ['phase_ml', 'therapeutic_area_ml', 'endpoint_bin'],
        ['phase_ml', 'therapeutic_area_ml'],
        ['phase_ml'],
    ],
}


def fit_groups(train: pd.DataFrame, ycol: str, levels: list[list[str]], min_n: int = MIN_N):
    global_median = train[ycol].median()
    fitted = []
    for keys in levels:
        if not keys:
            continue
        grouped = train.groupby(keys, dropna=False)[ycol].agg(['size', 'median']).reset_index()
        fitted.append((keys, grouped[grouped['size'] >= min_n].copy()))
    return global_median, fitted


def predict_one(row: pd.Series, global_median: float, fitted) -> float:
    for keys, grouped in fitted:
        matches = grouped
        for key in keys:
            matches = matches[matches[key].eq(row[key])]
        if not matches.empty:
            return float(matches.iloc[0]['median'])
    return float(global_median)


def cv_eval(mask: pd.Series, ycol: str, levels: list[list[str]], folds_n: int = 5) -> dict[str, float]:
    base = df.loc[mask].copy().reset_index(drop=True)
    rng = np.random.default_rng(123)
    folds = rng.integers(0, folds_n, len(base))
    preds = np.empty(len(base))
    actual = base[ycol].to_numpy(float)
    for fold in range(folds_n):
        train = base.iloc[folds != fold]
        test = base.iloc[folds == fold]
        global_median, fitted = fit_groups(train, ycol, levels)
        preds[folds == fold] = [predict_one(row, global_median, fitted) for _, row in test.iterrows()]
    return {
        'median_abs_month_error': round(float(np.median(np.abs(actual - preds))), 2),
        'mean_abs_month_error': round(float(np.mean(np.abs(actual - preds))), 2),
        'log_mae': round(float(np.mean(np.abs(np.log1p(actual) - np.log1p(preds)))), 4),
    }

rows = []
for label, mask, ycol in [
    ('total_completion', total_target, 'completion_duration_months'),
    ('primary_readout', primary_target, 'primary_completion_duration_months'),
]:
    for variant, levels in variants.items():
        rows.append({'target': label, 'variant': variant, **cv_eval(mask, ycol, levels)})

pd.DataFrame(rows).sort_values(['target', 'log_mae'])


In [ ]:
noncompleted = ~completed
summary_rows = []
for label, ycol, dtype_col in [
    ('total_estimated_noncompleted', 'completion_duration_months', 'completion_date_type'),
    ('primary_estimated_noncompleted', 'primary_completion_duration_months', 'primary_completion_date_type'),
]:
    mask = noncompleted & df[dtype_col].eq('ESTIMATED') & df[ycol].gt(0)
    s = df.loc[mask, ycol]
    summary_rows.append({
        'population': label,
        'n': int(len(s)),
        'p25': round(float(s.quantile(.25)), 2),
        'p50': round(float(s.quantile(.50)), 2),
        'p75': round(float(s.quantile(.75)), 2),
        'p90': round(float(s.quantile(.90)), 2),
    })

pd.DataFrame(summary_rows)


## Current Completed-Only Interpretation

Live-run results from 2026-06-03:

- Completed total-duration target: 20,476 rows; median 21.19 months; P25 11.99; P75 36.93; P90 58.64.
- Completed primary-readout target: 20,681 rows; median 18.37 months; P25 10.45; P75 30.49; P90 46.92.
- The existing clinical hierarchy has full fallback coverage through phase-only and keeps indication-level rows for about 73% of completed target rows.
- Exact modality refinement is too sparse at indication level for duration: only about 49% row coverage at `phase + indication + rare + modality` with `n >= 50`.
- Endpoint-duration bins are the strongest candidate variable by a large margin and improve 5-fold CV error when added to the clinical hierarchy.
- Therapeutic modality has signal, but adding a simple TA-modality fallback did not improve CV error over the clinical hierarchy.

Recommended D0 direction:

1. Use completed `ACTUAL` total duration as the benchmark target for `planned_duration_months`.
2. Build separate completed `ACTUAL` primary-readout percentiles for `planned_primary_completion_months` context.
3. Keep the current clinical hierarchy as the backbone.
4. Add coarse `primary_duration_months_ml` endpoint-duration bins as a duration-specific refinement/floor candidate, subject to a final policy decision.
5. Do not add therapeutic modality as a primary duration refinement in v1 unless a more constrained analysis proves incremental benefit.
6. Treat non-completed `ESTIMATED` dates as future planned-candidate/default-source evidence, not as historical benchmark targets for the first implementation.
